<a href="https://colab.research.google.com/github/Rakesh114166/Rakesh-dock/blob/main/win.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title **Install all the Required Software** { display-mode: "form" }
#@markdown Click the play button to compile system tools and the computational docking engines.
print("⏳ Step 1: Synchronizing operating system biophysics packages...")
!apt-get update -y > /dev/null && apt-get install -y openbabel > /dev/null
!pip install vina pandas py3Dmol --quiet
print("⚡ All backend calculation engines successfully installed!")

⏳ Step 1: Synchronizing operating system biophysics packages...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 54.8 MB/s eta 0:00:00
⚡ All backend calculation engines successfully installed!


In [2]:
#@title **Import Python modules** { display-mode: "form" }
#@markdown Click the play button to import standard data formatting and math frameworks into memory.
import os
import sys
import pandas as pd
from vina import Vina
import py3Dmol
print("✅ Core processing modules successfully loaded!")

✅ Core processing modules successfully loaded!


In [3]:
#@title **Create folder** { display-mode: "both" }
#@markdown Enter a **< Job Name >** without spaces on the right panel to map your project file layout.

Job_name = "1B6C" #@param {type:"string"}

# AUTOMATION FIX: Instantly strips away accidental trailing or leading spaces
Job_name_clean = Job_name.strip()

DIR = os.getcwd()
WRK_DIR = os.path.join(DIR, Job_name_clean)
PRT_FLD = os.path.join(WRK_DIR, "PROTEIN")
LIG_FLD = os.path.join(WRK_DIR, "LIGAND")
EXP_FLD = os.path.join(WRK_DIR, "EXPERIMENTAL")
DCK_FLD = os.path.join(WRK_DIR, "DOCKING")
INT_FLD = os.path.join(WRK_DIR, "INTERACTION")

folders = [WRK_DIR, PRT_FLD, LIG_FLD, EXP_FLD, DCK_FLD, INT_FLD]

print("📁 Generating data organization paths...")
for f in folders:
    if not os.path.exists(f):
        os.makedirs(f, exist_ok=True)
        print(f"   ✅ Created folder layout: {os.path.basename(f)}")
    else:
        print(f"   👉 Folder already exists: {os.path.basename(f)}")

📁 Generating data organization paths...
   ✅ Created folder layout: 1B6C
   ✅ Created folder layout: PROTEIN
   ✅ Created folder layout: LIGAND
   ✅ Created folder layout: EXPERIMENTAL
   ✅ Created folder layout: DOCKING
   ✅ Created folder layout: INTERACTION


In [4]:
#@title **Download & Prepare Receptor** { display-mode: "both" }
#@markdown Input your target protein ID code on the right panel to download and clean the target receptor.

Protein_PDB_ID = "1B6C" #@param {type:"string"}

import urllib.request
import subprocess
import os

# Force uppercase format and remove spaces to guarantee the database server locates the file
PDB_ID_CLEAN = Protein_PDB_ID.strip().upper()

# These variables rely on PRT_FLD, which was safely cleared of trailing spaces in Cell 3
pdb_path = os.path.join(PRT_FLD, f"{PDB_ID_CLEAN}.pdb")
receptor_pdbqt = os.path.join(PRT_FLD, f"{PDB_ID_CLEAN}_receptor.pdbqt")

print(f"⏳ Downloading structure {PDB_ID_CLEAN} from the RCSB PDB server...")

# Download stream using a browser identity header to bypass firewall blocks
url = f"https://files.rcsb.org/download/{PDB_ID_CLEAN}.pdb"
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})

try:
    with urllib.request.urlopen(req) as response, open(pdb_path, 'wb') as out_file:
        out_file.write(response.read())
    print("   ✅ PDB file downloaded successfully from RCSB.")
except Exception as e:
    print(f"   ⚠️ Primary database timed out: {e}. Trying secondary backup mirror (PDBe)...")
    fallback_url = f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{PDB_ID_CLEAN.lower()}.ent"
    try:
        fallback_req = urllib.request.Request(fallback_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(fallback_req) as response, open(pdb_path, 'wb') as out_file:
            out_file.write(response.read())
        print("   ✅ PDB file successfully recovered from PDBe backup server.")
    except Exception as fallback_err:
        print(f"   ❌ Secondary download mirror failed: {fallback_err}")

print("⏳ Converting model format (Adding explicit polar hydrogens, generating rigid receptor)...")
# Paths are safely wrapped in double quotes to satisfy the terminal interpreter layout
cmd = f'obabel -ipdb "{pdb_path}" -h -xr -opdbqt -O "{receptor_pdbqt}"'
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

# Final verification check
if os.path.exists(receptor_pdbqt) and os.path.getsize(receptor_pdbqt) > 500:
    print(f"🧬 Target protein clean and ready: {receptor_pdbqt}")
else:
    print("\n❌ Critical Error: Target receptor parsing failed.")
    print("📋 System Terminal Error Output:")
    print(result.stderr if result.stderr else "No terminal error logged. Verify Cell 1 was executed first.")

⏳ Downloading structure 1B6C from the RCSB PDB server...
   ✅ PDB file downloaded successfully from RCSB.
⏳ Converting model format (Adding explicit polar hydrogens, generating rigid receptor)...
🧬 Target protein clean and ready: /content/1B6C/PROTEIN/1B6C_receptor.pdbqt


In [5]:
#@title **Prepare Ligands from SMILES** { display-mode: "both" }
#@markdown Assign your compound identification labels and paste their structural SMILES keys on the right.

Ligand_1_Name = "Naringenin" #@param {type:"string"}
Ligand_1_SMILES = "O=C1CC(C2=CC=C(O)C=C2)OC3=CC(O)=CC(O)=C13" #@param {type:"string"}

Ligand_2_Name = "Chrysin" #@param {type:"string"}
Ligand_2_SMILES = "O=C1C=C(C2=CC=CC=C2)OC3=CC(O)=CC(O)=C13" #@param {type:"string"}

compounds = {}
if Ligand_1_Name.strip() and Ligand_1_SMILES.strip(): compounds[Ligand_1_Name.strip()] = Ligand_1_SMILES.strip()
if Ligand_2_Name.strip() and Ligand_2_SMILES.strip(): compounds[Ligand_2_Name.strip()] = Ligand_2_SMILES.strip()

print("⏳ Minimizing geometric conformations and building coordinate parameters...")
for name, smiles in compounds.items():
    output_ligand_pdbqt = os.path.join(LIG_FLD, f"{name}.pdbqt")
    # AUTOMATION FIX: Wrapped the output file path in explicit double quotes to protect spaces
    cmd = f'obabel -:"{smiles}" -O "{output_ligand_pdbqt}" --gen3d -h -p 7.4 --quiet'
    os.system(cmd)

    if os.path.exists(output_ligand_pdbqt):
        print(f"   ✅ Converted structure: {name}.pdbqt -> Saved to LIGAND folder")
    else:
        print(f"   ❌ Critical Error: Failed to generate structure file for {name}")

⏳ Minimizing geometric conformations and building coordinate parameters...
   ✅ Converted structure: Naringenin.pdbqt -> Saved to LIGAND folder
   ✅ Converted structure: Chrysin.pdbqt -> Saved to LIGAND folder


In [6]:
#@title **Run AutoDock Vina Docking Engine** { display-mode: "both" }
#@markdown Enter your pocket geometric coordinates and box dimensions on the right panel to execute calculations.

Center_X = -43.63 #@param {type:"number"}
Center_Y = 19.4 #@param {type:"number"}
Center_Z = -16.51 #@param {type:"number"}
Box_Size_Dimension = 25.0 #@param {type:"number"}

results_list = []
pocket_center = [Center_X, Center_Y, Center_Z]
pocket_size = [Box_Size_Dimension, Box_Size_Dimension, Box_Size_Dimension]

print("▶️ Initiating automated molecular docking calculation loops...")
for name in compounds.keys():
    ligand_pdbqt = os.path.join(LIG_FLD, f"{name}.pdbqt")
    output_poses = os.path.join(DCK_FLD, f"{name}_poses.pdbqt")

    # Safety verification check before initializing calculation frames
    if not os.path.exists(ligand_pdbqt):
        print(f"   ❌ Skipping compound {name}: '{ligand_pdbqt}' cannot be located. Please execute Cell 5 first.")
        continue

    try:
        v = Vina(sf_name='vina')
        v.set_receptor(receptor_pdbqt)
        v.set_ligand_from_file(ligand_pdbqt)
        v.compute_vina_maps(center=pocket_center, box_size=pocket_size)

        v.dock(exhaustiveness=8, n_poses=9)
        v.write_poses(output_poses, n_poses=9, overwrite=True)

        best_affinity = v.energies(n_poses=1)[0][0]
        results_list.append({"Compound ID": name, "Binding Affinity (kcal/mol)": best_affinity})
        print(f"   ✅ Calculation complete for {name}: {best_affinity} kcal/mol")
    except Exception as e:
        print(f"   ❌ Execution error for compound {name}: {e}")

# Compile, format, and display evaluation summary rankings
if results_list:
    df = pd.DataFrame(results_list).sort_values(by="Binding Affinity (kcal/mol)").reset_index(drop=True)
    print("\n==================================================")
    print("📊 FINAL RANKED VIRTUAL SCREENING RESULTS:")
    print("==================================================")
    display(df)

▶️ Initiating automated molecular docking calculation loops...
   ✅ Calculation complete for Naringenin: -5.077 kcal/mol
   ✅ Calculation complete for Chrysin: -5.468 kcal/mol

📊 FINAL RANKED VIRTUAL SCREENING RESULTS:


,Compound ID,Binding Affinity (kcal/mol)
0,Chrysin,-5.468
1,Naringenin,-5.077


In [8]:
#@title **Interactive 3D Structure Visualization** { display-mode: "both" }
#@markdown Select a ligand and rendering template on the right side panel to inspect structural bindings.

Visualized_Compound = "Naringenin" #@param ["Naringenin", "Chrysin"] {allow-input: true}
Protein_Rendering_Style = "cartoon" #@param ["cartoon", "stick", "sphere", "line"]

pdb_path = os.path.join(PRT_FLD, f"{Protein_PDB_ID}.pdb")
output_poses = os.path.join(DCK_FLD, f"{Visualized_Compound}_poses.pdbqt")
poses_pdb = os.path.join(DCK_FLD, f"{Visualized_Compound}_poses.pdb")

if os.path.exists(pdb_path) and os.path.exists(output_poses):
    os.system(f"obabel {output_poses} -O {poses_pdb} --quiet")

    viewer = py3Dmol.view(width=800, height=600)
    with open(pdb_path, "r") as f:
        viewer.addModel(f.read(), "pdb")
    viewer.setStyle({"model": 0}, {Protein_Rendering_Style: {"color": "spectrum"}})

    with open(poses_pdb, "r") as f:
        viewer.addModel(f.read(), "pdb")
    viewer.setStyle({"model": 1}, {"stick": {"colorscheme": "cyanCarbon"}})

    viewer.zoomTo({"model": 1})
    viewer.show()
    print(f"✨ Rendering 3D interactive layout for {Visualized_Compound} inside target protein pocket!")
else:
    print("❌ Data file missing. Make sure you ran Cell 3, 4, 5 and 6 successfully first.")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

✨ Rendering 3D interactive layout for Naringenin inside target protein pocket!
